# Davinci — Persian–English Tokenizer Design & Training

This Colab notebook prepares a reproducible SentencePiece tokenizer experiment for Davinci. It compares vocabulary sizes on the same corpus, measures Persian/English/code tokenization efficiency, checks Unicode round-tripping, and exports the selected model and an evaluation report.

**Important:** tokenizer choice is an experiment, not a predetermined answer. Vocabulary size and algorithm must be selected from measured compression, fertility, coverage, and downstream model constraints. This notebook does not claim that one setting is optimal.

## 1. Install dependencies

In [ ]:
!pip -q install "sentencepiece>=0.2.0,<0.3" "pandas>=2.2,<3" "matplotlib>=3.8,<4"

In [ ]:
import json, re, hashlib, unicodedata, shutil, importlib.metadata
from pathlib import Path
import pandas as pd
import sentencepiece as spm

from google.colab import drive
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Davinci/tokenizer")
CORPUS=ROOT/"corpus.txt"
OUT=ROOT/"models"
OUT.mkdir(parents=True,exist_ok=True)

def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## 2. Choose and inspect training inputs

Use cleaned, license-reviewed text from the earlier corpus notebook and reviewed SFT data. The default paths below are examples. Upload files or mount Drive if your files are elsewhere. Avoid training the tokenizer on validation/test text.

In [ ]:
INPUT_JSONL_PATHS = [
    "/content/drive/MyDrive/Davinci/sft_data/processed/farsinstruct_chat_sft.jsonl",
    "/content/drive/MyDrive/Davinci/data_pipeline/processed/persian_corpus_merged.jsonl",
]
# Explicit extraction rules are required for every input path.
INPUT_MODES = {
    "/content/davinci_sft/processed/farsinstruct_chat_sft.jsonl": {"mode":"messages"},
    "/content/davinci_data/processed/persian_corpus_merged.jsonl": {"mode":"text","field":"text"},
}
MAX_LINES_PER_FILE=500000

## 3. Normalize conservatively

Preserve Persian ZWNJ, punctuation, digits, English text, and code symbols. Normalize Unicode to NFC and map Arabic Yeh/Kaf to Persian forms. Do not strip diacritics or collapse meaningful punctuation without a measured reason.

In [ ]:
def normalize_text(text):
    text=unicodedata.normalize("NFC",str(text))
    text=text.replace(chr(0x064A),chr(0x06CC)).replace(chr(0x0643),chr(0x06A9))
    text=text.replace(chr(0x0640),"")
    bidi={0x200e,0x200f,*range(0x202a,0x202f),*range(0x2066,0x206a)}
    text="".join(ch for ch in text if ord(ch) not in bidi)
    text=re.sub("[ ]{2,}"," ",text)
    return text.strip()

def extract_record(obj,path):
    cfg=INPUT_MODES.get(path)
    if cfg is None: raise ValueError(f"Set INPUT_MODES explicitly for {path}")
    if cfg["mode"]=="messages":
        msgs=obj.get("messages")
        if not isinstance(msgs,list): return []
        return [m.get("content","") for m in msgs if isinstance(m,dict) and isinstance(m.get("content"),str)]
    if cfg["mode"]=="text":
        value=obj.get(cfg["field"])
        return [value] if isinstance(value,str) else []
    raise ValueError(f"Unsupported mode: {cfg['mode']}")

## 4. Build a deterministic tokenizer corpus

This stage reads JSONL inputs with explicit extraction rules. It records source-level counts and exact-line duplicates. It is not a replacement for document-level deduplication or a full privacy/license audit.

In [ ]:
if not INPUT_JSONL_PATHS: raise ValueError("Add at least one JSONL path to INPUT_JSONL_PATHS")
seen=set(); source_stats=[]; written=0; skipped=0
with CORPUS.open("w",encoding="utf-8") as out:
    for path_str in INPUT_JSONL_PATHS:
        path=Path(path_str)
        if not path.exists():
            raise FileNotFoundError(f"Configured tokenizer input does not exist: {path}")
        if path_str not in INPUT_MODES: raise ValueError(f"Add explicit INPUT_MODES entry for {path_str}")
        manifest_path=Path(INPUT_MANIFESTS[path_str])
        if not manifest_path.is_file():
            raise FileNotFoundError(f"Dataset manifest is missing: {manifest_path}")
        source_manifest=json.loads(manifest_path.read_text(encoding="utf-8"))
        if source_manifest.get("license_approved_for_use") is not True:
            raise PermissionError(f"Dataset license review is not approved for intended use: {path_str}")
        read_count=0; file_written=0
        with path.open(encoding="utf-8") as f:
            for line in f:
                if read_count>=MAX_LINES_PER_FILE: break
                read_count+=1
                try: obj=json.loads(line)
                except Exception:
                    skipped+=1; continue
                for raw in extract_record(obj,path_str):
                    text=normalize_text(raw)
                    if len(text)<2: skipped+=1; continue
                    digest=hashlib.sha256(text.encode("utf-8")).hexdigest()
                    if digest in seen: skipped+=1; continue
                    seen.add(digest)
                    out.write(text.replace("\n"," ")+"\n")
                    written+=1; file_written+=1
        source_stats.append({"path":path_str,"lines_read":read_count,"texts_written":file_written})
if written==0: raise ValueError("No training text was collected; mount or upload inputs and check paths.")
print({"unique_texts":written,"skipped":skipped,"corpus_bytes":CORPUS.stat().st_size})
print(pd.DataFrame(source_stats))
if written<1000: print("WARNING: small corpus; treat as a smoke test, not a final tokenizer.")

## 5. Inspect language and content coverage

In [ ]:
sample=[]
with CORPUS.open(encoding="utf-8") as f:
    for i,line in enumerate(f):
        if i>=20000: break
        sample.append(line.rstrip("\n"))
joined="\n".join(sample)
fa_chars=len(re.findall("[\u0600-\u06FF]",joined))
latin_chars=len(re.findall("[A-Za-z]",joined))
digit_chars=sum(ch.isdigit() for ch in joined)
print({"sample_lines":len(sample),"Persian_Arabic_block_chars":fa_chars,"Latin_chars":latin_chars,"digits":digit_chars})
for s in sample[:10]: print("-",s[:240])

## 6. Train comparable SentencePiece models

Train Unigram and BPE models with identical corpus and normalization settings. Vocabulary sizes are experimental candidates, not fixed requirements. Byte fallback improves coverage for unseen Unicode, at a vocabulary cost.

In [ ]:
VOCAB_SIZES=[16000,24000,32000]
ALGORITHMS=["unigram","bpe"]
SPECIAL_TOKENS=["<|system|>","<|user|>","<|assistant|>","<|tool|>","<|endofturn|>","<|tool_call|>","<|tool_result|>"]
model_paths=[]
for algo in ALGORITHMS:
    for vocab in VOCAB_SIZES:
        prefix=OUT/f"{algo}_{vocab}"
        spm.SentencePieceTrainer.train(
            input=str(CORPUS), model_prefix=str(prefix), model_type=algo,
            vocab_size=vocab, character_coverage=0.9995,
            byte_fallback=True, hard_vocab_limit=False,
            normalization_rule_name="identity",  # Corpus was normalized explicitly; preserve ZWNJ and punctuation.
            pad_id=0, unk_id=1, bos_id=2, eos_id=3,
            pad_piece="<pad>", unk_piece="<unk>", bos_piece="<s>", eos_piece="</s>",
            user_defined_symbols=SPECIAL_TOKENS,
            shuffle_input_sentence=False, input_sentence_size=0,
            num_threads=4, train_extremely_large_corpus=True)
        model_paths.append({"algorithm":algo,"requested_vocab":vocab,"model":str(prefix)+".model","vocab":str(prefix)+".vocab"})
print(pd.DataFrame(model_paths))

## 7. Evaluate tokenizer efficiency

Metrics are descriptive: tokens per whitespace-delimited word (fertility), characters per token, and encoded length. Lower token count is not automatically better; quality, vocabulary capacity, rare-word handling, and downstream model performance matter.

In [ ]:
EVAL_TEXTS={
 "Persian":"این یک نمونهٔ فارسی برای ارزیابی جداسازی واژه‌ها و نیم‌فاصله است.",
 "English":"This is an English sample for tokenizer evaluation and generalization.",
 "Mixed":"لطفاً این API را با Python و مدل‌های language modeling بررسی کن.",
 "Code":"def add(a, b):\n    return a + b\n",
 "Digits":"قیمت ۱۲۳۴۵ و 12345؛ نسخه v2.1 و HTTP/2",
 "ZWNJ":"می‌روم، کتاب‌ها، خانه‌ی بزرگ"}
def evaluate_model(model_path):
    proc=spm.SentencePieceProcessor(model_file=model_path)
    rows=[]
    for label,text in EVAL_TEXTS.items():
        ids=proc.encode(text,out_type=int)
        rows.append({"model":Path(model_path).stem,"sample":label,"tokens":len(ids),
                     "whitespace_words":max(1,len(text.split())),
                     "tokens_per_word":len(ids)/max(1,len(text.split())),
                     "chars_per_token":len(text)/max(1,len(ids)),
                     "roundtrip_ok":proc.decode(ids)==text})
    return rows
metrics=[]
for item in model_paths: metrics.extend(evaluate_model(item["model"]))
metrics_df=pd.DataFrame(metrics)
display(metrics_df.pivot(index="sample",columns="model",values="tokens"))
display(metrics_df.groupby("model")[["tokens_per_word","chars_per_token","roundtrip_ok"]].mean())

## 8. Check special-token IDs and Unicode round-trip

In [ ]:
checks=[]
for item in model_paths:
    proc=spm.SentencePieceProcessor(model_file=item["model"])
    special={tok:proc.piece_to_id(tok) for tok in SPECIAL_TOKENS}
    unk=proc.unk_id()
    roundtrip=all(proc.decode(proc.encode(t))==t for t in EVAL_TEXTS.values())
    checks.append({"model":Path(item["model"]).stem,"vocab_size":proc.get_piece_size(),
                   "special_tokens_present":all(v!=unk for v in special.values()),
                   "roundtrip_all_samples":roundtrip})
checks_df=pd.DataFrame(checks)
display(checks_df)
if not checks_df["special_tokens_present"].all():
    raise RuntimeError("At least one reserved special token resolves to <unk>; do not select/export these models.")
if not checks_df["roundtrip_all_samples"].all():
    raise RuntimeError("Unicode round-trip failed for at least one candidate; inspect normalization before selection.")

## 9. Choose the candidate explicitly

Do not automatically choose the model with the lowest token count. Select after reviewing results, corpus representativeness, model parameter budget, embedding/output matrix cost, and target training framework.

In [ ]:
SELECTED_MODEL=None  # Example: "unigram_24000"; choose after reviewing results.
if SELECTED_MODEL is None: raise ValueError("Set SELECTED_MODEL explicitly after reviewing evaluation results.")
selected=next((x for x in model_paths if Path(x["model"]).stem==SELECTED_MODEL),None)
if selected is None: raise ValueError("SELECTED_MODEL not found")
selected_model=Path(selected["model"])
print("Selected:",selected_model)

## 10. Save model card and evaluation artifacts

In [ ]:
REPORT=ROOT/"evaluation"
REPORT.mkdir(exist_ok=True)
metrics_df.to_csv(REPORT/"tokenizer_metrics.csv",index=False)
checks_df.to_csv(REPORT/"tokenizer_checks.csv",index=False)
shutil.copy2(selected_model,REPORT/"davinci_sentencepiece.model")
shutil.copy2(selected_model,ROOT/"davinci_sentencepiece.model")
shutil.copy2(selected_model.with_suffix(".vocab"),REPORT/"davinci_sentencepiece.vocab")
shutil.copy2(selected_model.with_suffix(".vocab"),ROOT/"davinci_sentencepiece.vocab")
proc=spm.SentencePieceProcessor(model_file=str(selected_model))
model_card={
 "model_name":"Davinci Persian-English SentencePiece tokenizer",
 "algorithm":selected["algorithm"],"requested_vocab_size":selected["requested_vocab"],
 "actual_vocab_size":int(proc.get_piece_size()),
 "normalization":"NFC; Arabic Yeh/Kaf mapped to Persian; tatweel and bidi controls removed; ZWNJ preserved",
 "byte_fallback":True,"special_tokens":SPECIAL_TOKENS,
 "special_token_ids":{tok:int(proc.piece_to_id(tok)) for tok in SPECIAL_TOKENS},
 "training_corpus":str(CORPUS),"training_text_count":written,
 "package_versions":{"sentencepiece":importlib.metadata.version("sentencepiece"),
                    "pandas":importlib.metadata.version("pandas"),
                    "matplotlib":importlib.metadata.version("matplotlib")},
 "normalization_rule_name":"identity",
 "corpus_sha256":sha256_file(CORPUS),
 "source_stats":source_stats,
 "input_manifests":{path:{"path":manifest,"sha256":hashlib.sha256(Path(manifest).read_bytes()).hexdigest()} for path,manifest in INPUT_MANIFESTS.items()},
 "training_parameters":{"character_coverage":0.9995,"byte_fallback":True,
                        "shuffle_input_sentence":False,"input_sentence_size":0,
                        "num_threads":4},
 "created_utc":pd.Timestamp.utcnow().isoformat(),
 "limitations":["Evaluation samples are small and manually authored.",
                "No downstream language-model loss or task evaluation was performed.",
                "Corpus license and privacy review must be completed independently."]}
(REPORT/"tokenizer_model_card.json").write_text(json.dumps(model_card,ensure_ascii=False,indent=2),encoding="utf-8")
print("Artifacts:",[p.name for p in REPORT.iterdir()])

## 11. Download artifacts

In [ ]:
from google.colab import files
for p in REPORT.iterdir():
    if p.is_file(): files.download(str(p))

## 12. Integration notes for Davinci

- SentencePiece model files are not automatically a Hugging Face PreTrainedTokenizerFast; integration requires a compatible tokenizer wrapper and explicit special-token mapping.
- Reserve token IDs consistently across pretraining, SFT, tool-call serialization, inference, and checkpoint metadata.
- Measure sequence lengths on representative Persian, English, mixed-language, and code corpora before finalizing context length and batch-size plans.
- A larger vocabulary reduces some sequence lengths but increases embedding/output parameters and can worsen rare morphology coverage. Compare total model budget, not tokenization alone.
- Retraining or changing the tokenizer after model pretraining generally requires embedding remapping or retraining; freeze the tokenizer before the main pretraining run.
- This notebook's evaluation is a sanity check. Add held-out real-world samples and downstream model evaluations before release.